# Description

Notebook to create a sin based dataset

In [ ]:
import os
import sys

CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim

# Locals
from utils import *
from dataset import Dataset
from models import *
from dlnr import DLNoiseReduction

In [ ]:
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [ ]:
def sin_function(x):
    """
    Function that takes in a value x and returns its sin.
    The values are scaled to be between 0 and 1.

    Args:
        x (float): value to be transformed.

    Returns:
        float: value of sin(x) scaled to be between 0 and 1.
    """

    return (1 + np.sin(np.pi * x))/2


def add_gaussian_noise(df, columns, mean=0, std=0.1):
    """
    Adds Gaussian noise to specified columns in a DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame to which noise will be added.
        columns (list): A list of column names in the DataFrame to which noise will be added.
        mean (int, optional): The mean of the Gaussian distribution. Defaults to 0.
        std (float, optional): The standard deviation of the Gaussian distribution. Defaults to 0.1.

    Returns:
        pandas.DataFrame: The DataFrame with added Gaussian noise to specified columns.
    """
    for col in columns:
        df[col] = df[col] + np.random.normal(loc=mean, scale=std, size=len(df))
    return df


In [ ]:
X_train = np.linspace(-4, 4, 10001)
y_train = sin_function(X_train)

df_train = pd.DataFrame({'X': X_train, 'y': y_train})
plt.scatter(df_train['X'], df_train['y'], color='b', s=0.1)
plt.legend(['f(x)=x^4 -x^3 -20x^2 -20*x +6'])

In [ ]:
X_val = np.linspace(-4, 4, 10001)
y_val = sin_function(X_val)

df_val = pd.DataFrame({'X': X_val, 'y': y_val})
plt.scatter(df_val['X'], df_val['y'], color='g', s=0.1)
plt.legend(['f(x)=x^4 -x^3 -20x^2 -20*x +6'])

In [ ]:
X_test = np.linspace(-4, 4, 10001)
y_test = sin_function(X_test)

df_test = pd.DataFrame({'X': X_test, 'y': y_test})
plt.scatter(df_test['X'], df_test['y'], color='r', s=0.1)
plt.legend(['f(x)=x^4 -x^3 -20x^2 -20*x +6'])

In [ ]:
sigma = 0.1
df_noisy = add_gaussian_noise(
        df=df_train.copy(),
        columns=[x for x in df_train.columns],
        mean=0,
        std=sigma
)

In [ ]:
plt.title('Data with noise')
plt.scatter(df_noisy['X'], df_noisy['y'], color='r', s=0.1)

In [ ]:
scaler = MinMaxScaler()
df_noisy = pd.DataFrame(scaler.fit_transform(df_noisy))
df_noisy.columns = ['X', 'y']

In [ ]:
df_train = pd.DataFrame(scaler.fit_transform(df_train))
df_train.columns = ['X', 'y']

In [ ]:
df_val = pd.DataFrame(scaler.transform(df_val))
df_val.columns = ['X', 'y']
df_val.describe().T

In [ ]:
df_test = pd.DataFrame(scaler.transform(df_test))
df_test.columns = ['X', 'y']
df_test.describe().T

In [ ]:
plt.title('Data with noise')
plt.scatter(df_noisy['X'], df_noisy['y'], color='r', s=0.1)

In [ ]:
# Transform the data into a tensor
bio_dataset_train = Dataset(
    input_df=df_noisy[['X']],
    target_df=df_noisy[['y']]
)
# Transform the data into a tensor
bio_dataset_val = Dataset(
    input_df=df_val[['X']],
    target_df=df_val[['y']]
)

# Create the dataloaders
batch_size = 64
train_dataloader = DataLoader(bio_dataset_train, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = DataLoader(bio_dataset_val, batch_size=batch_size, shuffle=True, num_workers=0)

In [ ]:
plt.title('Splitted data with noise')
plt.scatter(df_noisy['X'], df_noisy['y'], color='r', s=0.2, alpha=0.6)
plt.scatter(df_val['X'], df_val['y'], color='b', s=0.2, alpha=0.3)
plt.scatter(df_test['X'], df_test['y'], color='g', s=0.2, alpha=0.3)
plt.legend(['Train', 'Validation', 'Test'])

In [ ]:
model = GridFullyDenseNN(
    n_layers=2,
    hidden_layers=[
        (1, 1024),
        (1024, 1),
    ],
    dropout_layers=[
        0.0,
        0.0,
    ],
    activation_func_layers=[
        nn.ReLU(),
        nn.Identity(),
    ],
    want_dropout=[
        False,
        False,
    ],
    want_linear=[
        True,
        True,
    ],
    want_activation=[
        True,
        False,
    ],
)
model.to(device)

In [ ]:
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=model,
    train_generator=train_dataloader,
    val_generator=val_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=os.path.join(CHECKPOINT_PATH, f'model_sin_noise_{sigma}'),
)

In [ ]:
# model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)

In [ ]:
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_PATH, f'model_sin_noise_{sigma}.pth')))

In [ ]:
y_pred_train = model(torch.tensor(df_noisy['X'].values.reshape(-1, 1)).float().to(device)).cpu().detach().numpy().reshape(-1)
y_pred_val = model(torch.tensor(df_val['X'].values.reshape(-1, 1)).float().to(device)).cpu().detach().numpy().reshape(-1)
y_pred_test = model(torch.tensor(df_test['X'].values.reshape(-1, 1)).float().to(device)).cpu().detach().numpy().reshape(-1)

In [ ]:
mae = mean_absolute_error(df_test['y'].values, y_pred_test)
mse = mean_squared_error(df_test['y'].values, y_pred_test)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((df_test['y'].values - y_pred_test) / df_test['y'].values)) * 100
r_squared = r2_score(df_test['y'].values, y_pred_test)
print("MAE: ", mae)
print("MAPE: ", mape)
print("RMSE: ", rmse)
print("MSE: ", mse)
print("R-squared: ", r_squared)

In [ ]:
plt.scatter(df_noisy['X'].values, df_noisy['y'].values, color='b', label='True train', s=5, alpha=0.02)
# plt.scatter(df_noisy['X'].values, y_pred_train, color='cyan', label='Pred train', s=5, alpha=0.02)
# plt.scatter(df_val['X'].values, df_val['y'].values, color='r', label='True val', s=5, alpha=0.02)
# plt.scatter(df_val['X'].values, y_pred_val, color='pink', label='Pred val', s=5, alpha=0.02)
# plt.scatter(df_test['X'].values, df_test['y'].values, color='g', label='Datos con ruido', s=5, alpha=0.02)
plt.scatter(df_test['X'].values, y_pred_test, color='lime', label='Predicción', s=5, alpha=0.02)
plt.legend()
plt.show()

# Gradients to reduce noise

In [ ]:
df_no_noise = df_noisy.copy()
dlnr = DLNoiseReduction(model=model, criterion=criterion)
dlnr.fit(df_noisy['X'].values.reshape(-1, 1), df_noisy['y'].values.reshape(-1, 1))

In [ ]:
df_no_noise['X'], df_no_noise['y'] = dlnr.transform(
                                                nrr=0.02,
                                                nr_threshold=0.0,
                                                max_epochs=50,
                                                plot_progress=True
                                            )

In [ ]:
plt.scatter(df_train['X'].values, df_train['y'].values, color='b', label='Datos con ruido', s=5, alpha=0.02)
plt.scatter(df_no_noise['X'].values, df_no_noise['y'].values, color='r', label='Datos sin ruido', s=5, alpha=0.02)
plt.scatter(df_noisy['X'].values, df_noisy['y'].values, color='g', label='Datos con ruido', s=5, alpha=0.02)

In [ ]:
dlnr.assert_improvement(
    x_denoised = df_no_noise['X'].values.reshape(-1, 1),
    y_denoised = df_no_noise['y'].values.reshape(-1, 1),
    x_orig = df_train['X'].values.reshape(-1, 1),
    y_orig = df_train['y'].values.reshape(-1, 1)
)